In [ ]:
!pip install faiss-cpu pdfplumber newspaper3k sentence-transformers lxml_html_clean
!pip install pymupdf
!pip install "pillow==11.3.0" --force-reinstall --no-deps

In [ ]:
import os
import faiss
import pickle
import fitz
import numpy as np
import pickle
from newspaper import Article
from sentence_transformers import SentenceTransformer
from openai import OpenAI
EMBED_MODEL = "intfloat/multilingual-e5-large"
model = SentenceTransformer(EMBED_MODEL)
DATA_DIR = os.environ.get("DATA_DIR", "./data")
CHUNKS_PATH = os.path.join(DATA_DIR, "chunks.pkl")
INDEX_PATH =os.path.join(DATA_DIR, "faiss_index.bin")

In [ ]:
client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)

# =====================
# RETRIEVAL
# =====================

def retrieve(query, k=5):

    with open(CHUNKS_PATH, "rb") as f:
         chunks = pickle.load(f)
    index = faiss.read_index(INDEX_PATH)

    q_vector = model.encode(["query: " + query])

    q_vector = np.array(
        q_vector
    ).astype("float32")
    faiss.normalize_L2(q_vector)

    distances, indices = index.search(
        q_vector,
        k
    )

    results = []

    for idx in indices[0]:
        if idx == -1:
           continue
        results.append(
            chunks[idx]
        )

    return results
# =====================
# history recap
# =====================
def rewrite_query(question, history) :
    """
    history شكلها: [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]
    """
    if not history:
        return question

    history_text = "\n".join(
        f"{'المستخدم' if h['role']=='user' else 'المساعد'}: {h['content']}"
        for h in history[-4:]
    )

    rewrite_prompt = f"""بناءً على المحادثة السابقة، أعد صياغة السؤال الأخير ليكون سؤالًا مستقلًا وواضحًا بذاته، بدون حذف أي تفاصيل مهمة من السياق.

المحادثة السابقة:
{history_text}

السؤال الأخير: {question}

أعد كتابة السؤال الأخير فقط كسؤال مستقل، بدون أي شرح إضافي:"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": rewrite_prompt}]
    )
    return response.choices[0].message.content.strip()
# =====================
# ANSWER
# =====================

def answer(question ,  history):
    try:
        history = history or []
        standalone_question = rewrite_query(question, history)
        retrieved = retrieve(standalone_question)

        context = "\n\n".join(
        [r["text"] for r in retrieved]
         )

        prompt = f"""

        أنت مساعد قانوني متخصص في محاكم الأسرة المصرية.
        استخدم المعلومات فقط من السياق التالي:

        {context}

        السؤال:
        {question}

        أجب بالشكل التالي:

       📌 المشكلة

       ⚖️ الإجراءات المطلوبة

       📄 المستندات المطلوبة

       ⚠️ ملاحظات مهمة

       إذا لم تجد معلومة كافية داخل السياق
       قل:
       لا توجد معلومات كافية.
       """
        messages=[
            {
                "role": "system",
                "content":
                "أنت مساعد قانوني متخصص."
            }]
        messages.extend(history[-4:])
        messages.append({"role": "user", "content": prompt})
        response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages )

        return response.choices[0].message.content
    except Exception as e:

        return f"حدث خطأ: {str(e)}"
def rag_pipeline(question,history):
    return answer(question , history)
